# Satellite Analysis

## Download images

Docs:  
https://developers.google.com/maps/documentation/maps-static/start

## Square Grid

In [ ]:
import json
import numpy as np
import pandas as pd
import requests

from os import makedirs, path
from PIL import Image as PImage
from time import sleep

from env import GOOGLE_API_KEY

class obj(dict):
  __getattr__ = dict.get
  __setattr__ = dict.__setitem__
  __delattr__ = dict.__delitem__

def pad(n, l=3):
  return (l*"0" + str(n))[-l:] if n < 10**l else str(n)

def rnd(n, l=4):
  return float(round(n, l))

def to_obj(range_like):
  if "start" in range_like and "stop" in range_like:
    range_like = obj(range_like)
  if not (hasattr(range_like, "start") and hasattr(range_like, "start")):
    raise Exception("invalid parameter")
  return range_like

def range_step(range_like, l=4):
  range_like = to_obj(range_like)
  return rnd((range_like.stop - range_like.start) / range_like.count, l=l)

def range_space(range_like):
  range_like = to_obj(range_like)
  return np.linspace(range_like.start, range_like.stop, range_like.count, endpoint=True)

In [ ]:
MAPS_URL = "https://maps.googleapis.com/maps/api/staticmap"
IMG_SIZE = 512

MAPS_PARAMS = {
  "size": f"{IMG_SIZE}x{IMG_SIZE}",
  "scale": 1,
  "format": "jpg",
  "maptype": "satellite",
  "key": GOOGLE_API_KEY,
}

MAPS_HEADER = {
  "User-Agent": "Mozilla/5.0"
}

FOR = {
  "lat": { "start": -3.910, "stop": -3.710, "count": 64 }, # 32 OR 48 OR 64
  "lon": { "start": -38.655, "stop": -38.455, "count": 64 },
}

lat_cnt = FOR["lat"]["count"]
lon_cnt = FOR["lon"]["count"]
run_name = f"sat_{lon_cnt}x{lat_cnt}"

FOR["lat"]["step"] = range_step(FOR["lat"], 8)
FOR["lon"]["step"] = range_step(FOR["lon"], 8)
FOR["name"] = run_name

IMGS_DIR = "./satellites/imgs"
DATA_DIR = "./satellites/data"
SAT_IMGS_DIR = f"{IMGS_DIR}/{run_name}"

makedirs(DATA_DIR, exist_ok=True)
makedirs(IMGS_DIR, exist_ok=True)
makedirs(SAT_IMGS_DIR, exist_ok=True)

with open(f"{SAT_IMGS_DIR}/{run_name}.json", "w") as ofp:
  json.dump(FOR, ofp, indent=2, ensure_ascii=False)

In [ ]:
img_data = []
for y,lat_start in enumerate(range_space(FOR["lat"])):
  for x,lon_start in enumerate(range_space(FOR["lon"])):
    img_id = f"{pad(x, 2)}_{pad(y, 2)}"
    row = {
      "lat_center": rnd(lat_start + FOR["lat"]["step"] * 0.5, 5),
      "lon_center": rnd(lon_start + FOR["lon"]["step"] * 0.5, 5),
      "x": x,
      "y": y,
      "img_id": img_id
    }
    img_data.append(row)
    filepath = f"{SAT_IMGS_DIR}/{run_name}_{img_id}.jpg"

    if path.isfile(filepath):
      continue

    lat_stop = rnd(lat_start + FOR["lat"]["step"], 5)
    lon_stop = rnd(lon_start + FOR["lon"]["step"], 5)
    MAPS_PARAMS["visible"] = f"{lat_start},{lon_start}|{lat_stop},{lon_stop}"

    response = requests.get(MAPS_URL, headers=MAPS_HEADER, params=MAPS_PARAMS, stream=True)
    PImage.open(response.raw).save(filepath)
    sleep(0.25)

  pd.DataFrame(img_data).to_csv(f"{DATA_DIR}/{run_name}.csv", index=False)

### Combine Images

In [ ]:
nimgs = 64
simg = 100

img_prefix = f"sat_{nimgs}x{nimgs}"
img_dir = f"satellites/imgs/{img_prefix}"

fimg = PImage.new("RGB", (nimgs*simg, nimgs*simg))

for i in range(nimgs):
  ii = pad(i, 2)
  for j in range(nimgs):
    jj = pad(nimgs - j - 1, 2)
    img = PImage.open(f"{img_dir}/{img_prefix}_{ii}_{jj}.jpg").resize((simg, simg))
    fimg.paste(img, (i*simg, j*simg))

fimg.save(f"satellites/imgs/{img_prefix}.jpg")
display(fimg)

## Using Hex Map

In [ ]:
import json
import numpy as np
import requests

from os import makedirs, path
from PIL import Image as PImage
from time import sleep

from env import GOOGLE_API_KEY

def pad(n, l=3):
  return (l*"0" + str(n))[-l:] if n < 10**l else str(n)

In [ ]:
MAPS_URL = "https://maps.googleapis.com/maps/api/staticmap"
IMG_SIZE = 512

MAPS_PARAMS = {
  "size": f"{IMG_SIZE}x{IMG_SIZE}",
  "scale": 1,
  "format": "jpg",
  "maptype": "satellite",
  "key": GOOGLE_API_KEY,
}

MAPS_HEADER = {
  "User-Agent": "Mozilla/5.0"
}

run_name = f"sat_hex"

IMGS_DIR = "./satellites/imgs"
DATA_DIR = "./satellites/data"
SAT_IMGS_DIR = f"{IMGS_DIR}/{run_name}"

makedirs(DATA_DIR, exist_ok=True)
makedirs(IMGS_DIR, exist_ok=True)
makedirs(SAT_IMGS_DIR, exist_ok=True)

In [ ]:
with open("./satellites/data/fortaleza_renda_hex.geojson", "r") as ifp:
  hex_geo = sorted(json.load(ifp)["features"], key=lambda x: int(x["properties"]["id"]))

In [ ]:
img_data = []

for cnt,hx in enumerate(hex_geo):
  if cnt % 20 == 0:
    print(cnt, "/", len(hex_geo))

  hx_id = int(hx["properties"]["id"])
  img_id = f"{pad(hx_id, 5)}"

  coords = np.array(hx["geometry"]["coordinates"][0])
  coords_center = coords.mean(axis=0)
  coords_start = coords.min(axis=0) # bottom-left
  coords_stop = coords.max(axis=0) # top-right

  filepath = f"{SAT_IMGS_DIR}/{run_name}_{img_id}.jpg"

  if path.isfile(filepath):
    continue

  MAPS_PARAMS["visible"] = f"{coords_start[1]},{coords_start[0]}|{coords_stop[1]},{coords_stop[0]}"

  response = requests.get(MAPS_URL, headers=MAPS_HEADER, params=MAPS_PARAMS, stream=True)
  PImage.open(response.raw).save(filepath)
  sleep(0.25)

### Combine Images

In [ ]:
with open("./satellites/data/fortaleza_renda_hex.geojson", "r") as ifp:
  hex_geo = sorted(json.load(ifp)["features"], key=lambda x: int(x["properties"]["id"]))

coords = np.array([hx["geometry"]["coordinates"][0] for hx in hex_geo])

In [ ]:
# figure out how many images to use to tile

coords_min = coords.min(axis=0).min(axis=0)
coords_max = coords.max(axis=0).max(axis=0)
coords_diff = coords_max - coords_min

coords_avgs = coords.mean(axis=1)
coords_mins = coords.min(axis=1)
coords_maxs = coords.max(axis=1)

avg_diffs = np.concat((coords_maxs - coords_avgs, coords_avgs - coords_mins), axis=0).mean(axis=0)

In [ ]:
img_w = 6400
nimgs = (coords_diff / avg_diffs).astype(int)
hex_dim = int(img_w / max(nimgs) * 2.75)
img_h = int(img_w * nimgs[1] / nimgs[0] * 0.93)

img_prefix = f"sat_hex"
img_dir = f"satellites/imgs/{img_prefix}"

fimg = PImage.new("RGB", (img_w, img_h))

for cnt,hx in enumerate(hex_geo):
  hx_id = int(hx["properties"]["id"])
  img_id = f"{pad(hx_id, 5)}"
  img = PImage.open(f"{img_dir}/{img_prefix}_{img_id}.jpg").resize((hex_dim, hex_dim))

  coords = np.array(hx["geometry"]["coordinates"][0])
  coords_center = coords.mean(axis=0)
  coords_start = coords.min(axis=0) # bottom-left
  coords_stop = coords.max(axis=0) # top-right

  x = (coords_start[0] - coords_min[0]) / (coords_max[0] - coords_min[0]) * fimg.width
  y = fimg.height - (coords_stop[1] - coords_min[1]) / (coords_max[1] - coords_min[1]) * fimg.height

  fimg.paste(img, (int(x), int(y)))
  img.close()

fimg.save(f"satellites/imgs/{img_prefix}.jpg")
display(fimg)

## CV Analysis

### grid images and csv files

In [ ]:
!git clone https://github.com/direito-a-sombra/bus-view-satellites.git satellites

In [ ]:
import json
import pandas as pd

from os import makedirs, path
from PIL import Image as PImage

from torch import cuda, no_grad
from torchvision import transforms as T
from transformers import MaskFormerForInstanceSegmentation, MaskFormerImageProcessor

from env import HFTOKEN

IMG_RES = 32
VERSION = f"sat_{IMG_RES}x{IMG_RES}"
DATA_DIR = f"satellites/data"
IMGS_DIR = f"satellites/imgs/{VERSION}"

MODEL_ID = "thiagohersan/maskformer-satellite-trees"

In [ ]:
ade_mean = [ 0.485, 0.456, 0.406 ]
ade_std = [ 0.229, 0.224, 0.225 ]

ade_transform = T.Compose([
  T.ToTensor(),
  T.Normalize(mean=ade_mean, std=ade_std)
])

preprocessor = MaskFormerImageProcessor(
  do_resize=False,
  do_normalize=False,
  do_rescale=False,
  ignore_index=255,
  reduce_labels=False
)

device = "cuda" if cuda.is_available() else "cpu"
model = MaskFormerForInstanceSegmentation.from_pretrained(MODEL_ID, token=HFTOKEN).to(device)
included_labels = [l for l in model.config.id2label.values() if l not in ["other", "unknown"]]

In [ ]:
sat_df = pd.read_csv(f"{DATA_DIR}/{VERSION}.csv")
for l in included_labels:
  sat_df[f"{l}_pct"] = 0.0

In [ ]:
for idx,row in sat_df.iterrows():
  if idx % 16 == 0:
    print(idx)

  img_id = row["img_id"]

  img = PImage.open(f"{IMGS_DIR}/{VERSION}_{img_id}.jpg")
  iw,ih = img.size
  inputs = preprocessor(images=ade_transform(img), return_tensors="pt").to(device)

  with no_grad():
    outputs = model(**inputs)
    results = preprocessor.post_process_semantic_segmentation(outputs=outputs, target_sizes=[(ih, iw)])[0]

  label_ids, label_cnts = results.cpu().unique(return_counts=True)
  id2count = { int(lid): int(cnt) for lid, cnt in zip(label_ids, label_cnts) }

  for lid in label_ids.tolist():
    label = model.config.id2label[lid]
    if label in included_labels:
      sat_df.at[idx, f"{label}_pct"] = round(id2count[lid] / (iw * ih), 4)

In [ ]:
sat_df.to_csv(f"{DATA_DIR}/{VERSION}.1.csv", index=False)

## CV Analysis

### hex tiles from geojson file

In [ ]:
!git clone https://github.com/direito-a-sombra/bus-view-satellites.git satellites

In [ ]:
import json

from os import makedirs, path
from PIL import Image as PImage

from torch import cuda, no_grad
from torchvision import transforms as T
from transformers import MaskFormerForInstanceSegmentation, MaskFormerImageProcessor

from env import HFTOKEN

VERSION = f"sat_hex"
DATA_DIR = f"satellites/data"
IMGS_DIR = f"satellites/imgs/{VERSION}"

MODEL_ID = "thiagohersan/maskformer-satellite-trees"

In [ ]:
ade_mean = [ 0.485, 0.456, 0.406 ]
ade_std = [ 0.229, 0.224, 0.225 ]

ade_transform = T.Compose([
  T.ToTensor(),
  T.Normalize(mean=ade_mean, std=ade_std)
])

preprocessor = MaskFormerImageProcessor(
  do_resize=False,
  do_normalize=False,
  do_rescale=False,
  ignore_index=255,
  reduce_labels=False
)

device = "cuda" if cuda.is_available() else "cpu"
model = MaskFormerForInstanceSegmentation.from_pretrained(MODEL_ID, token=HFTOKEN).to(device)
included_labels = [l for l in model.config.id2label.values() if l not in ["other", "unknown"]]

In [ ]:
with open(f"{DATA_DIR}/fortaleza_renda_hex.geojson", "r") as ifp:
  hex_geo = json.load(ifp)

hex_geo["features"] = sorted(hex_geo["features"], key=lambda x: int(x["properties"]["id"]))

for hx in hex_geo["features"]:
  for l in included_labels:
    hx["properties"][f"{l}_pct"] = 0.0

In [ ]:
for cnt,hx in enumerate(hex_geo["features"]):
  hx_id = int(hx["properties"]["id"])
  img_id = f"{pad(hx_id, 5)}"

  img = PImage.open(f"{IMGS_DIR}/{VERSION}_{img_id}.jpg")
  iw,ih = img.size
  inputs = preprocessor(images=ade_transform(img), return_tensors="pt").to(device)

  with no_grad():
    outputs = model(**inputs)
    results = preprocessor.post_process_semantic_segmentation(outputs=outputs, target_sizes=[(ih, iw)])[0]

  label_ids, label_cnts = results.cpu().unique(return_counts=True)
  id2count = { int(lid): int(cnt) for lid, cnt in zip(label_ids, label_cnts) }

  for lid in label_ids.tolist():
    label = model.config.id2label[lid]
    if label in included_labels:
      hx["properties"][f"{label}_pct"] = round(id2count[lid] / (iw * ih), 4)

In [ ]:
with open(f"{DATA_DIR}/{VERSION}.geojson", "w") as ofp:
  json.dump(hex_geo, ofp, ensure_ascii=False)